### **TABL**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

from tqdm import tqdm 
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score
import torch
from torch.utils import data
import torch.nn as nn
import torch.optim as optim
import pickle
import torch.nn.functional as F
from torch.autograd import Variable
from torch.utils.data import DataLoader
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


ModuleNotFoundError: No module named 'pandas'

In [23]:
class Dataset(data.Dataset):
    """Characterizes a dataset for PyTorch"""
    def __init__(self, x, y, num_classes, dim):
        """Initialization""" 
        self.num_classes = num_classes
        self.dim = dim
        self.x = x   
        self.y = y

        self.length = x.shape[0] - (T/10) -self.dim + 1
        print(self.length)

        x = torch.from_numpy(x)
        self.x = torch.unsqueeze(x, 1)
        self.y = torch.from_numpy(y)

    def __len__(self):
        """Denotes the total number of samples"""
        return int(self.length)

    def __getitem__(self, i):
        input = self.x[i:i+self.dim, :]
        input = input.permute(1, 2, 0)
        input = torch.squeeze(input)

        return input, self.y[i]

In [33]:


def make_loaders_for_fold(fold,h=5,dim=10,selected_dimension=144,batch_size=256,train_dir="../data/training",test_dir="../data/testing",val_ratio=0.2):

    train_path = f"{train_dir}/Train_Dst_NoAuction_ZScore_CF_{fold}.txt"
    test_path  = f"{test_dir}/Test_Dst_NoAuction_ZScore_CF_{fold}.txt"

    dec_data = np.loadtxt(train_path)
    dec_test = np.loadtxt(test_path)

    n_cols = dec_data.shape[1]
    cut = int(n_cols * (1 - val_ratio))
    dec_train_raw = dec_data[:, :cut]
    dec_val_raw   = dec_data[:, cut:]

    y_train = dec_train_raw[-h, :].flatten()[dim-1:] - 1
    y_val   = dec_val_raw[-h, :].flatten()[dim-1:] - 1
    y_test  = dec_test[-h, :].flatten()[dim-1:] - 1


    X_train = dec_train_raw[:selected_dimension, :].T
    X_val   = dec_val_raw[:selected_dimension, :].T
    X_test  = dec_test[:selected_dimension, :].T


    ds_train = Dataset(X_train, y_train, 3, dim)
    ds_val   = Dataset(X_val,   y_val,   3, dim)
    ds_test  = Dataset(X_test,  y_test,  3, dim)

    train_loader = DataLoader(ds_train, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(ds_val,   batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(ds_test,  batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader, y_train

In [25]:
def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
def compute_class_weights(y_np):

    c0 = (y_np == 0).sum()
    c1 = (y_np == 1).sum()
    c2 = (y_np == 2).sum()
    w = torch.tensor([1e6/max(c0,1), 1e6/max(c1,1), 1e6/max(c2,1)], dtype=torch.float32, device=device)
    return w

In [ ]:
X, y = next(iter(train_loader))
print("X shape:", X.shape)
print("y shape:", y.shape)
print("X dtype:", X.dtype, "y dtype:", y.dtype)
print("y unique:", torch.unique(y)[:10])

X shape: torch.Size([256, 144, 10])
y shape: torch.Size([256])
X dtype: torch.float64 y dtype: torch.float64
y unique: tensor([0., 1., 2.], dtype=torch.float64)


In [26]:

def accuracy_from_logits(logits, y):
    preds = torch.argmax(logits, dim=1)
    return (preds == y).float().mean().item()

@torch.no_grad()
def eval_model(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_preds, all_true = [], []

    for X, y in loader:
        X = X.float().to(device)
        y = y.long().to(device)

        logits = model(X)
        loss = criterion(logits, y)
        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)
        all_preds.append(preds.detach().cpu().numpy())
        all_true.append(y.detach().cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_true  = np.concatenate(all_true)

    avg_loss = total_loss / len(loader)
    acc = (all_preds == all_true).mean()
    macro_f1 = f1_score(all_true, all_preds, average="macro")
    return avg_loss, acc, macro_f1

In [29]:
class MLPBaseline(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),              
            nn.Linear(144*10, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.net(x)

In [30]:
class CNNTimeBaseline(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(in_channels=144, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(in_channels=128, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        # x: (B,144,10)
        z = self.features(x)    
        z = z.mean(dim=2)             
        out = self.classifier(z)      
        return out

In [27]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0

    for X, y in loader:
        X = X.float().to(device)
        y = y.long().to(device)

        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [28]:
def fit(model, train_loader, val_loader, criterion, optimizer, epochs=20):
    best_val_f1 = -1.0
    best_state = None

    for epoch in range(1, epochs+1):
        tr_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        va_loss, va_acc, va_f1 = eval_model(model, val_loader, criterion)

        print(f"Epoch {epoch:03d} | train_loss={tr_loss:.4f} | val_loss={va_loss:.4f} | val_acc={va_acc:.4f} | val_macroF1={va_f1:.4f}")

        if va_f1 > best_val_f1:
            best_val_f1 = va_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return model

In [31]:
def run_9fold_cv(model_builder, model_name, epochs=20, lr=1e-3, batch_size=256, h=5, dim=10):
    set_seed(42)
    fold_results = []

    print(f"\n==============================")
    print(f"MODEL: {model_name}")
    print(f"==============================\n")

    for fold in range(1, 10):
        print(f"\n----- Fold {fold} -----")

        train_loader, val_loader, test_loader, y_train = make_loaders_for_fold(
            fold=fold, h=h, dim=dim, batch_size=batch_size
        )

        # weights from TRAIN only
        weights = compute_class_weights(y_train.astype(np.int64))
        criterion = nn.CrossEntropyLoss(weight=weights)

        model = model_builder().to(device)
        optimizer = optim.Adam(model.parameters(), lr=lr)

        model = fit(model, train_loader, val_loader, criterion, optimizer, epochs=epochs)

        te_loss, te_acc, te_f1 = eval_model(model, test_loader, criterion)
        print(f"Fold {fold} TEST | loss={te_loss:.4f} acc={te_acc:.4f} macroF1={te_f1:.4f}")

        fold_results.append({"fold": fold, "test_loss": te_loss, "test_acc": te_acc, "test_macroF1": te_f1})

    accs = [r["test_acc"] for r in fold_results]
    f1s  = [r["test_macroF1"] for r in fold_results]

    print(f"\n===== SUMMARY: {model_name} =====")
    print(f"Test Acc   mean={np.mean(accs):.4f} std={np.std(accs):.4f}")
    print(f"Test MacroF1 mean={np.mean(f1s):.4f} std={np.std(f1s):.4f}")

    return fold_results

In [35]:
class LSTMClassifier(nn.Module):
    def __init__(self, num_classes=3, hidden_size=128, num_layers=1, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=144,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.0 if num_layers == 1 else dropout
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)
        out, (h_n, c_n) = self.lstm(x)
        h_last = h_n[-1] 
        return self.head(h_last)

In [36]:
class TCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, dilation=1, dropout=0.2):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2  # برای حفظ طول
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size, padding=padding, dilation=dilation),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(out_ch, out_ch, kernel_size, padding=padding, dilation=dilation),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.res = nn.Conv1d(in_ch, out_ch, kernel_size=1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        return self.net(x) + self.res(x)

class TCNClassifier(nn.Module):
    def __init__(self, num_classes=3, dropout=0.2):
        super().__init__()
        self.tcn = nn.Sequential(
            TCNBlock(144, 128, kernel_size=3, dilation=1, dropout=dropout),
            TCNBlock(128, 128, kernel_size=3, dilation=2, dropout=dropout),
            TCNBlock(128, 128, kernel_size=3, dilation=4, dropout=dropout),
        )
        self.head = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        z = self.tcn(x)   
        z = z.mean(dim=2)      
        return self.head(z)

In [2]:
import pandas as pd
def summarize_cv(fold_results, model_name):
    df = pd.DataFrame(fold_results)
    summary = {
        "model": model_name,
        "acc_mean": df["test_acc"].mean(),
        "acc_std": df["test_acc"].std(ddof=0),
        "f1_mean": df["test_macroF1"].mean(),
        "f1_std": df["test_macroF1"].std(ddof=0),
    }
    return df, summary

ModuleNotFoundError: No module named 'pandas'

In [37]:
mlp_results = run_9fold_cv(
    model_builder=lambda: MLPBaseline(num_classes=3),
    model_name="MLP Baseline",
    epochs=20,
    lr=1e-3
)


MODEL: MLP Baseline


----- Fold 1 -----
31599.0
7893.0
38387.0
Epoch 001 | train_loss=1.0115 | val_loss=1.0664 | val_acc=0.6118 | val_macroF1=0.4760
Epoch 002 | train_loss=0.9106 | val_loss=0.9284 | val_acc=0.5951 | val_macroF1=0.5431
Epoch 003 | train_loss=0.8504 | val_loss=0.9455 | val_acc=0.5964 | val_macroF1=0.5446
Epoch 004 | train_loss=0.8094 | val_loss=0.9031 | val_acc=0.6227 | val_macroF1=0.5705
Epoch 005 | train_loss=0.7664 | val_loss=0.9249 | val_acc=0.5444 | val_macroF1=0.5178
Epoch 006 | train_loss=0.7308 | val_loss=1.0460 | val_acc=0.5173 | val_macroF1=0.4779
Epoch 007 | train_loss=0.6931 | val_loss=0.9185 | val_acc=0.6244 | val_macroF1=0.5668
Epoch 008 | train_loss=0.6626 | val_loss=0.9308 | val_acc=0.6683 | val_macroF1=0.5919
Epoch 009 | train_loss=0.6279 | val_loss=1.0398 | val_acc=0.6337 | val_macroF1=0.5612
Epoch 010 | train_loss=0.5904 | val_loss=0.9404 | val_acc=0.5977 | val_macroF1=0.5530
Epoch 011 | train_loss=0.5733 | val_loss=0.9648 | val_acc=0.6136 | val_macr

In [38]:
cnn_results = run_9fold_cv(
    model_builder=lambda: CNNTimeBaseline(num_classes=3),
    model_name="CNN Time Baseline",
    epochs=20,
    lr=1e-3
)


MODEL: CNN Time Baseline


----- Fold 1 -----
31599.0
7893.0
38387.0
Epoch 001 | train_loss=1.0329 | val_loss=1.0966 | val_acc=0.3706 | val_macroF1=0.3175
Epoch 002 | train_loss=0.9750 | val_loss=0.9855 | val_acc=0.4958 | val_macroF1=0.4677
Epoch 003 | train_loss=0.9259 | val_loss=0.9527 | val_acc=0.5568 | val_macroF1=0.5161
Epoch 004 | train_loss=0.8669 | val_loss=0.9169 | val_acc=0.5627 | val_macroF1=0.5279
Epoch 005 | train_loss=0.8015 | val_loss=0.8894 | val_acc=0.5922 | val_macroF1=0.5551
Epoch 006 | train_loss=0.7535 | val_loss=0.9030 | val_acc=0.6301 | val_macroF1=0.5801
Epoch 007 | train_loss=0.7113 | val_loss=0.8706 | val_acc=0.6745 | val_macroF1=0.6061
Epoch 008 | train_loss=0.6778 | val_loss=0.8903 | val_acc=0.6829 | val_macroF1=0.6146
Epoch 009 | train_loss=0.6433 | val_loss=0.8316 | val_acc=0.5870 | val_macroF1=0.5654
Epoch 010 | train_loss=0.6164 | val_loss=0.9066 | val_acc=0.6814 | val_macroF1=0.6165
Epoch 011 | train_loss=0.5847 | val_loss=0.9041 | val_acc=0.6810 | val

In [41]:
lstm_results = run_9fold_cv(
    model_builder=lambda: LSTMClassifier(num_classes=3, hidden_size=128),
    model_name="LSTM",
    epochs=20,
    lr=1e-3
)


MODEL: LSTM


----- Fold 1 -----
31599.0
7893.0
38387.0
Epoch 001 | train_loss=1.0032 | val_loss=0.9716 | val_acc=0.5960 | val_macroF1=0.5133
Epoch 002 | train_loss=0.8481 | val_loss=0.9112 | val_acc=0.5675 | val_macroF1=0.5258
Epoch 003 | train_loss=0.7551 | val_loss=0.8996 | val_acc=0.6223 | val_macroF1=0.5602
Epoch 004 | train_loss=0.6885 | val_loss=0.8720 | val_acc=0.5846 | val_macroF1=0.5556
Epoch 005 | train_loss=0.6297 | val_loss=0.8641 | val_acc=0.6461 | val_macroF1=0.5980
Epoch 006 | train_loss=0.5685 | val_loss=0.8940 | val_acc=0.5619 | val_macroF1=0.5417
Epoch 007 | train_loss=0.5207 | val_loss=0.9328 | val_acc=0.5867 | val_macroF1=0.5593
Epoch 008 | train_loss=0.4656 | val_loss=0.9629 | val_acc=0.5763 | val_macroF1=0.5489
Epoch 009 | train_loss=0.4179 | val_loss=0.9358 | val_acc=0.5867 | val_macroF1=0.5596
Epoch 010 | train_loss=0.3688 | val_loss=1.0113 | val_acc=0.5175 | val_macroF1=0.5076
Epoch 011 | train_loss=0.3334 | val_loss=1.0489 | val_acc=0.6621 | val_macroF1=0.60

In [40]:
tcn_results = run_9fold_cv(
    model_builder=lambda: TCNClassifier(num_classes=3),
    model_name="TCN",
    epochs=20,
    lr=1e-3
)



MODEL: TCN


----- Fold 1 -----
31599.0
7893.0
38387.0
Epoch 001 | train_loss=1.0210 | val_loss=0.9800 | val_acc=0.4864 | val_macroF1=0.4646
Epoch 002 | train_loss=0.9191 | val_loss=0.9558 | val_acc=0.5300 | val_macroF1=0.4840
Epoch 003 | train_loss=0.8455 | val_loss=0.8658 | val_acc=0.6358 | val_macroF1=0.5832
Epoch 004 | train_loss=0.7877 | val_loss=0.8706 | val_acc=0.5160 | val_macroF1=0.5115
Epoch 005 | train_loss=0.7488 | val_loss=0.8953 | val_acc=0.5309 | val_macroF1=0.5215
Epoch 006 | train_loss=0.7170 | val_loss=0.8409 | val_acc=0.6107 | val_macroF1=0.5816
Epoch 007 | train_loss=0.6844 | val_loss=0.8663 | val_acc=0.6497 | val_macroF1=0.5991
Epoch 008 | train_loss=0.6578 | val_loss=0.8625 | val_acc=0.6632 | val_macroF1=0.6114
Epoch 009 | train_loss=0.6366 | val_loss=0.8440 | val_acc=0.6608 | val_macroF1=0.6113
Epoch 010 | train_loss=0.6123 | val_loss=0.8525 | val_acc=0.6535 | val_macroF1=0.6061
Epoch 011 | train_loss=0.5882 | val_loss=0.8307 | val_acc=0.6304 | val_macroF1=0.597

In [1]:
df_mlp, sum_mlp = summarize_cv(mlp_results, "MLP")
df_cnn, sum_cnn = summarize_cv(cnn_results, "CNN")
df_lstm, sum_lstm = summarize_cv(lstm_results, "LSTM")
df_tcn, sum_tcn = summarize_cv(tcn_results, "TCN")

NameError: name 'summarize_cv' is not defined

In [ ]:

summary_df = pd.DataFrame([sum_mlp, sum_cnn, sum_lstm, sum_tcn]).sort_values("f1_mean", ascending=False)
summary_df